<a href="https://colab.research.google.com/github/charlesanthony1996/explainable-ai-assignment1/blob/main/lab_track_a_backup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1 - load the uci dataset

In [ ]:
!pip install lime numpy pandas torch

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=58662933ddf39b6a89b44e322a1c38d7db38b04b002ac8b0fcf792152d30f726
  Stored in directory: /root/.cache/pip/wheels/7c/04/5c/157dc9106512a6c7a30653ec064490c94a49e0fc8f63d19ab9
Successfully built lime


In [ ]:
import os
import glob
import copy
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

from lime.lime_tabular import LimeTabularExplainer

In [ ]:
 # to show wide tables in the terminal
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

seed = 42

np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available else "cpu")

In [ ]:
data_url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
zip_path = "online_retail_ii.zip"

if len(glob.glob("*.xlsx")) == 0:
  print("downloading the dataset...")

  urllib.request.urlretrieve(data_url, zip_path)

  with zipfile.ZipFile(zip_path) as zip_file:
    zip_file.extractall(".")

downloading the dataset...


In [ ]:
# find the excel file that came out of the zip
excel_files = glob.glob("*.xlsx")
print(excel_files)

excel_path = excel_files[0]

['online_retail_II.xlsx']


In [ ]:
# the excel file has two sheets
sheets = pd.read_excel(excel_path, sheet_name=None)
sheet_names = list(sheets.keys())

print(sheet_names)

first_sheet = sheets[sheet_names[0]]
second_sheet = sheets[sheet_names[1]]

['Year 2009-2010', 'Year 2010-2011']


In [ ]:
# remove the overlapping days from the first sheet
second_sheet_start = second_sheet["InvoiceDate"].min()

first_sheet = first_sheet[first_sheet["InvoiceDate"] < second_sheet_start]

raw = pd.concat(
    [first_sheet, second_sheet],
    ignore_index=True
)

print("rows: ", raw.shape[0])
print("From: ", raw["InvoiceDate"].min())
print("To: ", raw["InvoiceDate"].max())


print(raw.head())

rows:  1044848
From:  2009-12-01 07:45:00
To:  2011-12-09 12:50:00
  Invoice StockCode                          Description  Quantity         InvoiceDate  Price  Customer ID         Country
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12 2009-12-01 07:45:00   6.95      13085.0  United Kingdom
1  489434    79323P                   PINK CHERRY LIGHTS        12 2009-12-01 07:45:00   6.75      13085.0  United Kingdom
2  489434    79323W                  WHITE CHERRY LIGHTS        12 2009-12-01 07:45:00   6.75      13085.0  United Kingdom
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48 2009-12-01 07:45:00   2.10      13085.0  United Kingdom
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24 2009-12-01 07:45:00   1.25      13085.0  United Kingdom
